# AE2: carga y exploración descriptiva del dataset

**Necesidad:** crear un informe con el que las personas puedan perfilar los barrios de una ciudad y presupuestar la compra o el arriendo de un inmueble, como base para plantear su búsqueda.

**Objetivos específicos**
1. Conocer por barrio el costo típico de arriendo y de venta.
2. Filtrar por barrio el costo y cuántas ofertas hay de arriendo y de venta.
3. Identificar los barrios con más ofertas.

**Dataset:** [`aptologia/arriendo-rentabilidad-por-barrio-colombia-2026`](https://huggingface.co/datasets/aptologia/arriendo-rentabilidad-por-barrio-colombia-2026), configuración `rentabilidad_arriendo_venta_barrio`, split `train`. Autor: Aptología. Licencia CC BY 4.0. Es público: no requiere token.

Este notebook:
1. Carga el dataset desde Hugging Face en un DataFrame de pandas.
2. Hace una revisión básica: registros, filas y columnas, tipos y nulos.
3. Explica qué representa cada fila y qué variables respaldan la necesidad.
4. Genera automáticamente `AE2/resultados/resumen_dataframe.txt`.

## 1. Preparar el entorno e importar la clase de carga

Se buscan los directorios padres hasta encontrar `src/clases_po` (la clase de carga de la Actividad 1) y se agrega `src` a `sys.path`. Así el notebook funciona sin importar desde qué carpeta se abra.

In [1]:
import sys  # Permite modificar las rutas donde Python busca módulos
from pathlib import Path  # Manejo de rutas de carpetas y archivos
from datetime import datetime  # Para registrar la fecha de generación del resumen

ROOT = Path.cwd().resolve()  # Carpeta desde la que se ejecuta el notebook
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():  # Subimos hasta encontrar src/clases_po
    ROOT = ROOT.parent  # Una carpeta hacia arriba

sys.path.insert(0, str(ROOT / "src"))  # Agregamos src/ a las rutas de búsqueda de Python

import pandas as pd  # Librería para trabajar con DataFrames
from clases_po import HFDatasetLoader  # Clase reutilizable para cargar datasets de Hugging Face

print("Raíz del proyecto:", ROOT)  # Confirmamos la raíz encontrada

/Users/sanmartin/Documents/IUDdeA/Proyecto Integrado 3/PI3_actividad1/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: /Users/sanmartin/Documents/IUDdeA/Proyecto Integrado 3/PI3_actividad1


## 2. Configurar y cargar el dataset

El dataset tiene dos configuraciones; usamos `rentabilidad_arriendo_venta_barrio`, que es la tabla de arriendo y venta por barrio. Se carga **completo** (sin muestra).

In [2]:
DATASET_ID = "aptologia/arriendo-rentabilidad-por-barrio-colombia-2026"  # Identificador del dataset en Hugging Face
CONFIG = "rentabilidad_arriendo_venta_barrio"  # Configuración (subconjunto) a cargar
SPLIT = "train"  # División a cargar
MAX_FILAS = None  # None = dataset completo; un número = solo una muestra de esas filas

loader = HFDatasetLoader(  # Creamos el cargador
    dataset_name=DATASET_ID,  # Qué dataset
    config=CONFIG,  # Qué configuración
    split=SPLIT,  # Qué división
    max_rows=MAX_FILAS,  # Límite de filas (None = todas)
)
df = loader.load()  # Descargamos y convertimos a DataFrame
print(loader.info())  # Resumen: dataset, configuración, split, filas y columnas

Generating train split: 100%|██████████| 979/979 [00:00<00:00, 30608.74 examples/s]

{'dataset': 'aptologia/arriendo-rentabilidad-por-barrio-colombia-2026', 'config': 'rentabilidad_arriendo_venta_barrio', 'split': 'train', 'filas': 979, 'columnas': 19}


## 3. Revisión básica

### 3.1 Algunos registros

In [3]:
df.head(10)  # Primeras 10 filas del DataFrame

,fecha_arriendo,fecha_venta,ciudad_id,ciudad,barrio_id,barrio,avisos_arriendo,mediana_arriendo,avisos_venta,mediana_venta,ok_todos,rent_bruta_todos_pct_calc,avisos_2_arriendo,mediana_2_arriendo,avisos_2_venta,mediana_2_venta,ok_2alc,rent_bruta_2alc_pct_calc,anios_arriendo_2alc_calc
0,2026-10-05,2026-10-05,medellin,Medellín,laureles,Laureles,6722,4000000,4067,791000000,True,6.068268,1385,3400000.0,441,6.250000e+08,True,6.528000,15.318627
1,2026-10-05,2026-10-05,medellin,Medellín,castropol,Castropol,2788,5350000,666,980000000,True,6.551020,828,4550000.0,140,9.375000e+08,True,5.824000,17.170330
2,2026-10-05,2026-10-05,medellin,Medellín,el-tesoro,El Tesoro,2680,7500000,719,1300000000,True,6.923077,545,6600000.0,140,1.160000e+09,True,6.827586,14.646465
3,2026-10-05,2026-10-05,medellin,Medellín,santa-maria-de-los-angeles,Santa Maria de los Angeles,2321,5500000,649,1050000000,True,6.285714,679,4700000.0,93,7.950000e+08,True,7.094340,14.095745
4,2026-10-05,2026-10-05,medellin,Medellín,calasanz,Calasanz,2281,2600000,968,440000000,True,7.090909,788,2400000.0,293,3.550000e+08,True,8.112676,12.326389
5,2026-10-05,2026-10-05,medellin,Medellín,ciudad-del-rio,Ciudad del Rio,2008,4500000,511,790000000,True,6.835443,564,4500000.0,145,8.700000e+08,True,6.206897,16.111111
6,2026-10-05,2026-10-05,medellin,Medellín,los-balsos,Los Balsos,2002,6000000,286,1375000000,True,5.236364,470,5800000.0,47,1.180000e+09,True,5.898305,16.954023
7,2026-10-05,2026-10-05,medellin,Medellín,las-palmas,Las Palmas,1941,4300000,672,945000000,True,5.460317,838,4350000.0,233,8.800000e+08,True,5.931818,16.858238
8,2026-10-05,2026-10-05,medellin,Medellín,altos-del-poblado,Altos del Poblado,1825,5500000,616,1100000000,True,6.000000,640,5000000.0,226,9.500000e+08,True,6.315789,15.833333
9,2026-10-05,2026-10-05,medellin,Medellín,san-lucas,San Lucas,1617,6700000,458,1350000000,True,5.955556,439,7500000.0,109,1.315000e+09,True,6.844106,14.611111


### 3.2 Cantidad de filas y columnas

In [4]:
filas, columnas = df.shape  # shape devuelve (filas, columnas)
print(f"Filas: {filas}")  # Cantidad de filas
print(f"Columnas: {columnas}")  # Cantidad de columnas

Filas: 979
Columnas: 19


### 3.3 Nombres, tipos de columnas y valores nulos

In [5]:
estructura = pd.DataFrame({  # Tabla con una fila por columna del dataset
    "tipo": df.dtypes.astype(str),  # Tipo de dato de cada columna
    "nulos": df.isna().sum(),  # Cantidad de valores nulos
    "% nulos": (df.isna().mean() * 100).round(1),  # Porcentaje de nulos
})
estructura  # Mostramos la tabla

,tipo,nulos,% nulos
fecha_arriendo,str,0,0.0
fecha_venta,str,0,0.0
ciudad_id,str,0,0.0
ciudad,str,0,0.0
barrio_id,str,0,0.0
barrio,str,0,0.0
avisos_arriendo,int64,0,0.0
mediana_arriendo,int64,0,0.0
avisos_venta,int64,0,0.0
mediana_venta,int64,0,0.0


### 3.4 Cobertura por ciudad
Cuántos barrios tiene cada ciudad; sirve para saber en qué ciudades el informe tendrá más detalle.

In [6]:
df["ciudad"].value_counts()  # Número de barrios (filas) por ciudad

ciudad
Bogotá          333
Medellín        324
Cali            115
Bucaramanga      52
Barranquilla     47
Rionegro         33
Chía             23
Cartagena        19
Cúcuta           17
Madrid            6
Soacha            4
Manizales         4
Pereira           2
Name: count, dtype: int64

### 3.5 Comprobación del significado de las columnas calculadas
La ficha del dataset no explica todas las columnas, así que comprobamos una: si `rent_bruta_todos_pct_calc` es la rentabilidad bruta anual, debe ser igual a `mediana_arriendo × 12 ÷ mediana_venta × 100`.

In [7]:
rent_calculada = df["mediana_arriendo"] * 12 / df["mediana_venta"] * 100  # Arriendo de un año como % del precio de venta
diferencia = (rent_calculada - df["rent_bruta_todos_pct_calc"]).abs().max()  # Mayor diferencia frente a la columna del dataset
print(f"Diferencia máxima: {diferencia:.6f}")  # Si es ~0, la interpretación es correcta

Diferencia máxima: 0.000000


## 4. Qué representa cada fila y qué variables respaldan la necesidad

**Cada fila es un barrio de una ciudad colombiana**, con el resumen de los avisos de inmuebles publicados en portales en la fecha de corte (`fecha_arriendo` y `fecha_venta`, octubre de 2026): cuántos avisos hay de arriendo y de venta, y su precio mediano. No son inmuebles individuales, sino agregados por barrio.

| Objetivo | Variables que lo respaldan |
|---|---|
| Costo por barrio (arriendo y venta) | `barrio`, `ciudad`, `mediana_arriendo` (COP/mes), `mediana_venta` (COP) |
| Filtrar por barrio el costo y cuántos negocios hay | `barrio`, `ciudad`, `avisos_arriendo`, `avisos_venta` |
| Barrios con más ofertas | `avisos_arriendo + avisos_venta` |
| Presupuestar el negocio (¿comprar o arrendar?) | `rent_bruta_todos_pct_calc` (rentabilidad bruta anual, %), `anios_arriendo_2alc_calc` (años de arriendo que equivalen al precio de compra) |
| Afinar por tamaño del inmueble | columnas `*_2_*` y `*_2alc*` (inmuebles de 2 alcobas, según su nombre) |
| Confiabilidad del dato | `ok_todos`, `ok_2alc` (marcan si el barrio cumple el criterio de calidad del autor) |

**Notas:**
- El dataset trae la **mediana**, no el promedio. Para precios inmobiliarios es más adecuada, porque unos pocos inmuebles de lujo no la distorsionan.
- Las columnas `*_2alc` tienen valores nulos en barrios con pocos avisos de 2 alcobas; para el análisis principal se usan las columnas generales.
- El significado de `ok_*` y `2alc` se infiere de los nombres y los datos; la rentabilidad se comprobó en la sección 3.5.

## 5. Generar automáticamente el resumen en .txt

Se genera a partir del mismo `df` revisado arriba. La carpeta se crea si no existe y el archivo se sobrescribe en cada ejecución.

In [8]:
RUTA_RESUMEN = ROOT / "AE2" / "resultados" / "resumen_dataframe.txt"  # Ruta relativa a la raíz del proyecto


def generar_resumen(df: pd.DataFrame) -> str:  # Arma el texto del resumen a partir del DataFrame
    filas, columnas = df.shape  # Dimensiones del DataFrame
    muestra = "No (dataset completo)" if MAX_FILAS is None else f"Sí, primeras {MAX_FILAS} filas"  # ¿Se usó muestra?
    lineas = [  # Líneas del resumen, en orden
        "RESUMEN DEL DATAFRAME",  # Título
        "=" * 60,  # Separador
        f"Generado: {datetime.now():%Y-%m-%d %H:%M:%S}",  # Fecha y hora de generación
        f"Dataset: {DATASET_ID}",  # Identificador del dataset
        f"URL: https://huggingface.co/datasets/{DATASET_ID}",  # Enlace al dataset
        f"Configuración: {CONFIG}",  # Configuración usada
        f"Split: {SPLIT}",  # Split usado
        f"Muestra: {muestra}",  # Indica si se usó una muestra
        "",  # Línea en blanco
        f"Filas: {filas}",  # Cantidad de filas
        f"Columnas: {columnas}",  # Cantidad de columnas
        "",  # Línea en blanco
        "Columnas y tipos:",  # Encabezado
        *[f"- {col}: {tipo}" for col, tipo in df.dtypes.items()],  # Una línea por columna con su tipo
        "",  # Línea en blanco
        "Valores nulos por columna:",  # Encabezado
        *[f"- {col}: {int(n)}" for col, n in df.isna().sum().items()],  # Una línea por columna con sus nulos
        f"Total de valores nulos: {int(df.isna().sum().sum())}",  # Total general
    ]
    return "\n".join(lineas) + "\n"  # Unimos todo con saltos de línea


RUTA_RESUMEN.parent.mkdir(parents=True, exist_ok=True)  # Creamos AE2/resultados si no existe
RUTA_RESUMEN.write_text(generar_resumen(df), encoding="utf-8")  # Escribimos (o sobrescribimos) el archivo en UTF-8
print("Resumen guardado en:", RUTA_RESUMEN.relative_to(ROOT))  # Mostramos la ruta relativa

Resumen guardado en: AE2/resultados/resumen_dataframe.txt


### Verificación del archivo generado

In [9]:
contenido = RUTA_RESUMEN.read_text(encoding="utf-8")  # Leemos el archivo recién creado
assert DATASET_ID in contenido  # Verificamos que incluya el identificador del dataset
assert f"Filas: {len(df)}" in contenido  # Verificamos que las filas coincidan con el DataFrame revisado
assert f"Columnas: {df.shape[1]}" in contenido  # Verificamos que las columnas coincidan
print(contenido)  # Mostramos el contenido completo

RESUMEN DEL DATAFRAME
Generado: 2026-10-25 23:04:01
Dataset: aptologia/arriendo-rentabilidad-por-barrio-colombia-2026
URL: https://huggingface.co/datasets/aptologia/arriendo-rentabilidad-por-barrio-colombia-2026
Configuración: rentabilidad_arriendo_venta_barrio
Split: train
Muestra: No (dataset completo)

Filas: 979
Columnas: 19

Columnas y tipos:
- fecha_arriendo: str
- fecha_venta: str
- ciudad_id: str
- ciudad: str
- barrio_id: str
- barrio: str
- avisos_arriendo: int64
- mediana_arriendo: int64
- avisos_venta: int64
- mediana_venta: int64
- ok_todos: bool
- rent_bruta_todos_pct_calc: float64
- avisos_2_arriendo: int64
- mediana_2_arriendo: float64
- avisos_2_venta: int64
- mediana_2_venta: float64
- ok_2alc: bool
- rent_bruta_2alc_pct_calc: float64
- anios_arriendo_2alc_calc: float64

Valores nulos por columna:
- fecha_arriendo: 0
- fecha_venta: 0
- ciudad_id: 0
- ciudad: 0
- barrio_id: 0
- barrio: 0
- avisos_arriendo: 0
- mediana_arriendo: 0
- avisos_venta: 0
- mediana_venta: 0
- 